# UrbanPulse — Notebook 01: EDA & Data Cleaning

Bible §5 Notebook 01. All logic lives in the repo-root modules (`cleaning.py`, `eda.py`, `io_utils.py`); this notebook is the narrative layer that calls them, so there is no duplicated logic.

**Produces:** `data/cleaned.parquet` + the EDA tables/plots in `reports/eda/`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import cleaning, eda, io_utils, config
import pandas as pd
pd.set_option('display.max_columns', 40)

## 1. Load raw data

In [2]:
raw = io_utils.load_raw()
print(raw.shape)
raw.head(3)

(266112, 34)


,TIMEINT,date,LINK_ID,DAY,VEHS(ALL)_1,SPEEDAVGARITH(ALL)_1,SPEEDAVGHARM(ALL)_1,QUEUEDELAY(ALL)_1,OCCUPRATE(ALL)_1,VEHS(ALL)_2,SPEEDAVGARITH(ALL)_2,SPEEDAVGHARM(ALL)_2,QUEUEDELAY(ALL)_2,OCCUPRATE(ALL)_2,VEHS(ALL)_3,SPEEDAVGARITH(ALL)_3,SPEEDAVGHARM(ALL)_3,QUEUEDELAY(ALL)_3,OCCUPRATE(ALL)_3,VEHS(ALL)_4,SPEEDAVGARITH(ALL)_4,SPEEDAVGHARM(ALL)_4,QUEUEDELAY(ALL)_4,OCCUPRATE(ALL)_4,VEHS(ALL)_5,SPEEDAVGARITH(ALL)_5,SPEEDAVGHARM(ALL)_5,QUEUEDELAY(ALL)_5,OCCUPRATE(ALL)_5,VEHS(ALL)_6,SPEEDAVGARITH(ALL)_6,SPEEDAVGHARM(ALL)_6,QUEUEDELAY(ALL)_6,OCCUPRATE(ALL)_6
0,900-1200,2024-07-01 0:00,1,1,228,146.24,136.25,247.13,0.3064,196.0,159.29,133.63,174.48,0.7921,228.0,160.45,126.87,146.59,1.0195,222.0,159.16,126.28,206.08,1.1821,198.0,155.52,134.94,179.73,0.7288,3.0,15.91,8.64,95.40,0.2005
1,1200-1500,2024-07-01 0:05,1,1,223,159.91,152.15,229.73,0.2840,215.0,149.31,121.20,227.76,1.7137,237.0,157.25,111.37,233.93,1.3930,238.0,173.75,149.21,230.36,0.6739,198.0,170.08,148.28,200.49,0.3336,2.0,17.71,13.93,13.05,0.0991
2,1500-1800,2024-07-01 0:10,1,1,218,153.16,143.45,245.56,0.2909,205.0,163.11,131.81,196.27,0.7721,242.0,172.53,133.67,204.92,1.1739,240.0,174.86,149.89,184.62,0.8511,187.0,156.51,139.76,254.97,0.2590,7.0,22.75,17.16,62.96,0.5037


## 2. Data quality — occupancy exceedance (measured on RAW, pre-cap)
Sensor saturation pushes occupancy above 1.0 under gridlock (Bible §2 Issue 1).

In [3]:
eda.occupancy_exceedance(raw)

,lane,rows_over_1,max_value
0,1,1484,1.6573
1,2,16321,3.0201
2,3,20373,2.9352
3,4,15544,2.8665
4,5,10842,2.9486
5,6,0,0.8469


## 3. Clean (9 mandatory steps)
Note the Lane 6 speed deviation documented in `cleaning.scale_speeds`: lanes 1-5 are 0.1 km/h (÷10), Lane 6 is already km/h.

In [4]:
df = cleaning.clean(raw)
df.shape

(266112, 41)

## 4. Missing values & dtypes

In [5]:
eda.missing_report(df)

,column,dtype,missing
0,date,datetime64[us],0
1,LINK_ID,int64,0
2,DAY,int64,0
3,VEHS(ALL)_1,int64,0
4,SPEEDAVGARITH(ALL)_1,float64,0
5,SPEEDAVGHARM(ALL)_1,float64,0
6,QUEUEDELAY(ALL)_1,float64,0
7,OCCUPRATE(ALL)_1,float64,0
8,VEHS(ALL)_2,int64,0
9,SPEEDAVGARITH(ALL)_2,float64,0


## 5. Per-link congestion (ranked by mean queue) — top 10
Cross-check vs Bible: Link 37 chronic (highest mean queue), Link 36 the critical hotspot by congested %, Link 16 ≈6.2%, Link 5 ≈3.9%.

In [6]:
eda.per_link_congestion(df).head(10)

,LINK_ID,mean_queue_s,mean_occup,mean_speed_kmh,congested_pct
0,37,380.620317,0.419444,20.074592,5.63
1,36,366.488648,0.620818,21.417093,15.20
2,42,330.757428,0.285741,17.073558,0.67
3,16,328.730436,0.552874,24.880755,6.20
4,35,323.495578,0.315301,21.860265,0.07
5,38,312.939153,0.158416,21.067177,0.00
6,30,311.204554,0.456105,17.443594,2.28
7,5,308.961216,0.841456,23.972673,3.87
8,41,305.603667,0.345638,21.062486,0.00
9,53,300.409130,0.241869,20.541593,0.00


## 6. Lane 6 — the ghost lane
≈25.5% active, ≈41 km/h when running: the unused capacity the ECHO counterfactual will exploit.

In [7]:
eda.lane6_analysis(df)

,active_rate,zero_rate,mean_speed_active_kmh,mean_speed_inactive_kmh
0,0.2546,0.7454,41.18561,0.0


## 7. Hourly & daily patterns

In [8]:
eda.hourly_patterns(df)

,hour,mean_vehs,mean_queue_s,mean_occup
0,0,690.599567,215.437248,0.322864
1,1,689.142947,214.577491,0.321044
2,2,687.151335,213.037504,0.320044
3,3,684.209867,214.117894,0.316313
4,4,688.401786,213.662754,0.315468
5,5,685.096050,213.375539,0.318899
6,6,687.478355,215.018098,0.319006
7,7,687.994589,213.172366,0.317230
8,8,1150.924152,246.637248,0.442109
9,9,1160.927309,257.700731,0.444447


In [9]:
eda.daily_patterns(df)

,day_number,mean_vehs,mean_queue_s
0,1,746.167982,220.347588
1,2,741.246370,218.571262
2,3,741.968382,217.396758
3,4,744.943287,218.837416
4,5,741.462489,216.695091
5,6,743.559396,218.280852
6,7,740.804135,217.669421
7,8,744.638626,218.644466
8,9,740.985901,217.709631
9,10,745.473432,217.532513


## 8. Correlation structure
Note the counterintuitive **positive** speed↔volume correlation (≈+0.68).

In [10]:
eda.correlation_matrix(df).round(2)

,hour,total_vehs,mean_occup,mean_queue_s,mean_speed_kmh,congested
hour,1.00,-0.00,0.00,-0.01,0.01,-0.02
total_vehs,-0.00,1.00,0.79,0.24,0.68,0.08
mean_occup,0.00,0.79,1.00,0.33,0.60,0.12
mean_queue_s,-0.01,0.24,0.33,1.00,0.66,0.21
mean_speed_kmh,0.01,0.68,0.60,0.66,1.00,0.07
congested,-0.02,0.08,0.12,0.21,0.07,1.00


## 9. Plots (saved to reports/eda/)

In [11]:
eda.plot_metric_distributions(df, config.EDA_REPORTS_DIR)
eda.plot_hourly(df, config.EDA_REPORTS_DIR)
eda.plot_correlation(df, config.EDA_REPORTS_DIR)
print('plots written')

plots written


## 10. Integrity gate + save

In [12]:
import json
print(json.dumps(cleaning.integrity_report(df), indent=2))
io_utils.save_parquet(df, config.CLEANED_PARQUET)
print('saved', config.CLEANED_PARQUET)

{
  "rows": 266112,
  "rows_match_expected": true,
  "links": 66,
  "missing_cells": 0,
  "max_occupancy": 1.0,
  "timeint_dropped": true,
  "has_lane6_active": true,
  "has_stall_flags": true,
  "speed_min_kmh": 0.0,
  "speed_max_kmh": 41.618
}


saved /home/claude/urbanpulse/data/cleaned.parquet
